# 1. Create the slice (once)

Creates the slice described by `NODES` in [deployment_config.py](deployment_config.py) — one VM per
Kubernetes node, including **every** agent used by any method (`aggregator` and `authority` are only used by
`fed-encrypt-dynamic`, but they are kept so all four methods can run on the same slice).

Each site gets its own FABNetv4 network; routes between the sites are added so all VMs can reach each other.
Rerunning this notebook is safe: an existing slice is reused and networking is only (re)applied where missing.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run

fablib = fablib_manager()

for spec in cfg.NODES:
    print(f"{spec['name']:<12} {spec['site']:<5} {spec['cores']:>2} cores {spec['ram']:>3} GB RAM {spec['disk']:>4} GB disk  {spec.get('host', '')}")

## Create (or reuse) the slice

Takes 10–20 minutes. If submission fails because a pinned worker (`host`) has no capacity, remove/change `host`
in `deployment_config.py`, delete the failed slice in **99_slice_management** and rerun.

In [ ]:
existing = [s.get_name() for s in fablib.get_slices()]
if cfg.SLICE_NAME in existing:
    slice_ = fh.get_slice(fablib)
    print(f"Reusing existing slice {cfg.SLICE_NAME} ({slice_.get_state()})")
else:
    slice_ = fh.create_slice(fablib)

slice_.list_nodes();

## Configure the data-plane network

Also rerun this after rebooting any VM (addresses are not persistent).

In [ ]:
fh.configure_networking(slice_)

## Verify connectivity from `control` to every VM

In [ ]:
control = fh.control_node(slice_)
for node in slice_.get_nodes():
    if node.get_name() == cfg.CONTROL_NODE:
        continue
    ip = fh.data_ip(node)
    ok = run(control, f"ping -c 2 -W 2 {ip} > /dev/null && echo ok || echo FAILED", quiet=True).strip()
    print(f"{node.get_name():<12} {ip:<16} {ok}")

In [ ]:
fh.print_ssh_commands(slice_)